# Correction First model
In this correction, I will remake the previous model version 5

In [2]:
# Libraries to use
import json
import statistics
import msprime
import numpy as np
import matplotlib.pyplot as plt

In [15]:
# ---- SIMULATION------

GEN_TIME = 29

# Fixed values for dates and rates
def years_to_gens(years):
    return int(years / GEN_TIME)
 
T_OOA = years_to_gens(70_000)
T_EAS_BER = years_to_gens(40_000)
T_BOTTLENECK_START = years_to_gens(25_000)
#T_NAT_FOUND = years_to_gens(16_000)  # This is no longer used since from NAT to BER it doesn't change much
T_NNA_SAM = years_to_gens(14_000)
 
MIGRATION_RATE = 1e-5

# ----- Demography Object -----
# Function to define a fresh demography object for every run. 
# Needed because msprime objects accumulate state so calls would not be independent.
def build_demography():
    demography = msprime.Demography()
    demography.add_population(name="SAM", initial_size=600, initially_active=True)
    demography.add_population(name="NNA", initial_size=6_000)
    demography.add_population(name="EAS", initial_size=10_000)
    demography.add_population(name="AFR", initial_size=40_000, initially_active=True)

    # Add demographic events
    demography.add_population_split(time=T_NNA_SAM, derived=["NNA"], ancestral="SAM") # Ancestor NAT where SAM and NNA come from
    demography.add_population_parameters_change(time=T_NNA_SAM, population="SAM", initial_size=2_000) # Size of NAT after bottleneck
    demography.add_population_parameters_change(time=T_BOTTLENECK_START, population="SAM", initial_size=800) # Beringia bottleneck
    demography.add_population_split(time=T_EAS_BER, derived=["EAS"], ancestral="SAM")
    demography.add_population_parameters_change(time=T_EAS_BER, population="SAM", initial_size=2_000) #OOA population of Ne=2000
    demography.add_population_split(time=T_OOA, derived=["SAM"], ancestral="AFR")
    demography.add_population_parameters_change(time=T_OOA, population="AFR", initial_size=10_000)

    # Add migration between populations
    demography.set_symmetric_migration_rate(["NNA", "SAM"], rate=MIGRATION_RATE)
    demography.add_migration_rate_change(time=T_EAS_BER, source="SAM", dest="AFR", rate=MIGRATION_RATE)
    demography.add_migration_rate_change(time=T_EAS_BER, source="AFR", dest="SAM", rate=MIGRATION_RATE)
    
    # Sanity check
    demography.sort_events()
    return demography

# Check the debug
print(build_demography().debug())



DemographyDebugger
╠════════════════════════════════╗
║ Epoch[0]: [0, 482) generations ║
╠════════════════════════════════╝
╟    Populations (total=4 active=4)
║    ┌─────────────────────────────────────────────────────────────────────┐
║    │     │     start│       end│growth_rate  │  SAM  │  NNA  │ EAS │ AFR │
║    ├─────────────────────────────────────────────────────────────────────┤
║    │  SAM│     600.0│     600.0│ 0           │   0   │ 1e-05 │  0  │  0  │
║    │  NNA│    6000.0│    6000.0│ 0           │ 1e-05 │   0   │  0  │  0  │
║    │  EAS│   10000.0│   10000.0│ 0           │   0   │   0   │  0  │  0  │
║    │  AFR│   40000.0│   40000.0│ 0           │   0   │   0   │  0  │  0  │
║    └─────────────────────────────────────────────────────────────────────┘
╟    Events @ generation 482
║    ┌──────────────────────────────────────────────────────────────────────────────┐
║    │  time│type        │parameters         │effect                                │
║    ├─────────────────

In [ ]:
# ------------------------------------------------------------------
# ---------------------- Replications ------------------------------
# ------------------------------------------------------------------
N = 30

stats_names = [
    "diversity_AFR", "diversity_EAS", "diversity_NNA", "diversity_SAM",
    "TajimasD_AFR", "TajimasD_EAS", "TajimasD_NNA", "TajimasD_SAM",
    #"Dxy_NNA-SAM", "Dxy_EAS-NNA", "Dxy_EAS-SAM", "Dxy_AFR-NNA", "Dxy_AFR-SAM", "Dxy_AFR-EAS",
    #"Fst_NNA-SAM", "Fst_EAS-NNA", "Fst_EAS-SAM", "Fst_AFR-NNA", "Fst_AFR-SAM", "Fst_AFR-EAS"    
]

# Collect statistics values across replicates
replicate_values = {stat:[] for stat in stats_names}

for seed in range(1, N+1):   # N replications with a different seed each
    # Build demography
    demography = build_demography()
    # Create simulation
    ts = msprime.sim_ancestry(
        samples={"NNA": 50, "SAM": 50, "EAS": 50, "AFR": 50},
        demography=demography,
        sequence_length=1_000_000,
        random_seed=seed
    )

    # Add mutations. Mutation seed also changing. Use different seed so ts and mts don't share a stream accidentally 
    mts = msprime.sim_mutations(ts, rate=1.25e-8, random_seed=seed + 100)

    pop_ids = {name: [p.id for p in demography.populations if p.name == name][0]   # ex: {'SAM': 0}
               for name in ["AFR", "EAS", "NNA", "SAM"]}

    samples = {name: mts.samples(population=pid) for name, pid in pop_ids.items()}  # ex: {'AFR': [0, 1..., 99], 'SAM': [100,...199],...}

    for name, s in samples.items():
        # Diversity (pi)
        replicate_values[f"diversity_{name}"].append(mts.diversity(sample_sets=s))
        # Tajima's D
        replicate_values[f"TajimasD_{name}"].append(mts.Tajimas_D(sample_sets=s))




In [17]:
# ------- Statistics ---------
summary = {}
for name, values in replicate_values.items():
    summary[f"{name}_mean"] = statistics.mean(values)
    summary[f"{name}_std"] = statistics.stdev(values)

print(f"\n=== Summary statistics: v5, mean +/- std across {N} replicates ===")
print("-- Nucleotide diversity (pi) --")
for name in ["AFR", "EAS", "NNA", "SAM"]:
    m, s = summary[f"diversity_{name}_mean"], summary[f"diversity_{name}_std"]
    print(f"{name}:{m:.6f} +/- {s:.6f}")

print("-- Tajima's D --")
for name in ["AFR", "EAS", "NNA", "SAM"]:
    m, s = summary[f"TajimasD_{name}_mean"], summary[f"TajimasD_{name}_std"]
    print(f"  {name}: {m:.4f} +/- {s:.4f}")


=== Summary statistics: v5, mean +/- std across 30 replicates ===
-- Nucleotide diversity (pi) --
AFR:0.000561 +/- 0.000273
EAS:0.000431 +/- 0.000280
NNA:0.000276 +/- 0.000207
SAM:0.000179 +/- 0.000291
-- Tajima's D --
  AFR: -1.0149 +/- 0.7207
  EAS: -0.0894 +/- 1.0877
  NNA: 0.2387 +/- 1.4407
  SAM: 0.6934 +/- 1.7077


## Korrigiert? NOCH NICHT

### Adding population growth rate

Project: evolutionary shifts in pre- and post-contact Native American
populations (ancient-DNA / msprime demographic modeling).

WHAT THIS SCRIPT TESTS
A bottleneck that is never followed by recovery mathematically produces a
POSITIVE Tajima's D (the population loses rare variants by drift, and
nothing replaces them) and a site-frequency spectrum (SFS) with relatively
FEWER rare/low-frequency variants than an unbottlenecked population. That
was shown explicitly in the previous script: EAS and AMR, each
modeled as a simple bottleneck held flat forever, came out with strongly
positive Tajima's D (1.35-2.39 depending on run).

Real non-African human populations do show some of that bottleneck
signature, but it is routinely offset in the data by what actually
happened next: rapid population growth after the founding bottleneck
(the "serial founder + growth" model is standard in human population
genetics; growth floods a population with new, young, low-frequency
mutations, which pulls Tajima's D back down and restores rare variants to
the SFS).

This script builds the same simple AFR (trunk) -> EAS -> AMR topology
twice - once with NO growth (bottleneck held flat, same as the earlier
script) and once WITH literature-anchored exponential growth after each
split - and compares diversity, Tajima's D, and the folded SFS between
them, replicated 20x each so the comparison isn't resting on a single
random seed.

HOW GROWTH IS PARAMETERIZED (the convention from the user-provided
Gutenkunst et al. 2009-style example)
msprime's `growth_rate` is defined so that `initial_size` is the
population's size AT TIME 0 (the present), and going backward in time its
size shrinks as:
    N(t) = initial_size * exp(-growth_rate * t)
So if we know a population's FOUNDING size N_found at the moment it
splits off (time T in the past) and we want a specific PRESENT-DAY size
N_now, the growth rate that connects them is:
    growth_rate = ln(N_now / N_found) / T
    initial_size = N_now   (NOT N_found - this is the detail that trips
                             people up; you give msprime the end point,
                             not the starting point)
This is exactly the pattern in the Gutenkunst-style reference example the
user supplied for CEU/CHB, and it's what's used below for EAS and AMR.

PARAMETER SOURCES
  - generation_time = 29 years                      (used throughout this project since v1)
  - T_AFR_EAS = 40,000 years ~ 1,379 generations     (project's established Out-of-Africa / East Asia split time)
  - T_EAS_AMR = 16,000 years ~ 551 generations       (project's established Asia -> Americas split time)
  - N_AFR = 40,000, flat, no growth                  (v3)
  - N_EAS founding = 2,000 (bottleneck size, v1/v3); present-day = 10,000 (v3's established EAS size)
        -> growth_rate r_EAS = ln(10000/2000) / 1379 ~ 0.001167 /generation
  - N_AMR founding = 1,791 (Rider et al. 2024 Native American ancestral
    size, used as NAT's size in v5); present-day = 17,910, i.e. a 10x
    growth factor, following Hey (2005) "On the Number of New World
    Founders" (PLoS Biology), which reports a founding Ne of "about 70"
    growing "by a factor of about 10" to a New World Ne of "less than
    1,000" - we don't reuse Hey's absolute numbers (they're for a
    different, much smaller model), only the ~10x growth FACTOR, applied
    to this project's own founding size.
        -> growth_rate r_AMR = ln(17910/1791) / 551 ~ 0.004171 /generation

MIGRATION, AND AN IMPORTANT ENGINE LIMITATION FOUND WHILE BUILDING THIS
The user asked for AFR<->EAS migration while EAS still exists (i.e.
before the EAS->AMR split). This project has already documented one
msprime edge case (v3's README): `set_symmetric_migration_rate()` can
crash with "Attempt to move a lineage into an inactive population" when
migration is active "from time 0" between a long-lived population and one
that later merges away via a split, once sample sizes get large. That fix
(pin migration to the right activation time) was built for ghost
populations that activate PARTWAY through the model.

Here, both AFR and EAS are sampled, present-day populations, active from
time 0 - the v3 situation doesn't apply directly. But testing this
specific topology (AFR trunk, EAS splitting off, migration between them)
turned up a related, genuinely reproducible instance of the SAME class of
engine issue: at the project's standard migration rate of 1e-5, this
exact 2-3 population split+migration structure crashes for roughly
15-40% of random seeds (confirmed in a 30-100 seed sweep in this
project's scratch testing), regardless of the split time, buffer zeroing,
or whether migration is set via set_symmetric_migration_rate() or by
writing directly into the migration matrix. Lowering the migration rate
removed the crashes entirely in that same seed sweep (0/100 failures at
1e-7, versus 8/100 at 1e-6 and 13/30 at 1e-5). So migration here is set
to a conservative 1e-7/generation - weak (on the order of ~0.001-0.004
expected migrants per generation given these population sizes) but
numerically safe. This is a real limitation of the simulation engine at
this msprime version (1.4.4) worth flagging if later versions are used
and a stronger rate is wanted - it would be worth re-testing whether a
newer msprime release has fixed this.



In [21]:
"""
================================================================================
 v6 - POST-BOTTLENECK EXPONENTIAL GROWTH (AFR trunk / EAS / AMR)
================================================================================
"""

import json
import math
import statistics

import msprime
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------------
# 1. Parameters
# ------------------------------------------------------------------------
GEN_TIME = 29

T_AFR_EAS = round(40_000 / GEN_TIME)   # 1379 generations
T_EAS_AMR = round(16_000 / GEN_TIME)   # 551 generations

N_AFR = 40_000                         # flat, no growth 

N_EAS_FOUND = 2_000                    # size right after the OOA/EAS bottleneck
N_EAS_NOW = 10_000                     # present-day EAS size (v3)
R_EAS = math.log(N_EAS_NOW / N_EAS_FOUND) / T_AFR_EAS

N_AMR_FOUND = 1_791                    # Rider et al. 2024 founding size
N_AMR_NOW = 17_910                     # 10x growth factor (Hey et al. 2005)
R_AMR = math.log(N_AMR_NOW / N_AMR_FOUND) / T_EAS_AMR

MIG_RATE = 1e-7                        # see docstring: conservative, to avoid
                                        # a reproducible msprime crash at this
                                        # split topology and larger rates
SEQ_LEN = 1e6
RECOMB_RATE = 1e-8
MUT_RATE = 1.2e-8
N_SAMPLES = 50
N_REPLICATES = 20                      # cheap at this scale (~0.2s/replicate)


# ------------------------------------------------------------------------
# 2. Build the demography, with or without post-bottleneck growth
# ------------------------------------------------------------------------
def build_demography(with_growth):
    d = msprime.Demography()
    d.add_population(name="AFR", description="African (trunk population)",
                      initial_size=N_AFR, initially_active=True)

    if with_growth:
        # initial_size is the PRESENT-DAY size; growth_rate carries it
        # backward to the founding size at the split time (see docstring).
        d.add_population(name="EAS", description="East Asian",
                          initial_size=N_EAS_NOW, growth_rate=R_EAS, initially_active=True)
        d.add_population(name="AMR", description="American",
                          initial_size=N_AMR_NOW, growth_rate=R_AMR, initially_active=True)
    else:
        # Baseline: bottleneck size held flat forever (no recovery) - the
        # same structure as native_american_demography_simple_trunk_fixed.py.
        d.add_population(name="EAS", description="East Asian",
                          initial_size=N_EAS_FOUND, initially_active=True)
        d.add_population(name="AMR", description="American",
                          initial_size=N_AMR_FOUND, initially_active=True)

    # EAS splits off from AFR; AMR later splits off from EAS. Both AFR and
    # EAS are sampled/active from time 0, so migration between them needs
    # no special activation-time handling - just the rate-limit note above.
    d.add_population_split(time=T_EAS_AMR, derived=["AMR"], ancestral="EAS")
    d.add_population_split(time=T_AFR_EAS, derived=["EAS"], ancestral="AFR")
    d.set_symmetric_migration_rate(["AFR", "EAS"], rate=MIG_RATE)

    d.sort_events()
    return d


# ------------------------------------------------------------------------
# 3. Run N_REPLICATES independent simulations and collect statistics
# ------------------------------------------------------------------------
def run_replicates(with_growth):
    d = build_demography(with_growth)
    pops = ["AFR", "EAS", "AMR"]
    pop_ids = {p: [pp.id for pp in d.populations if pp.name == p][0] for p in pops}

    pi_vals = {p: [] for p in pops}
    tajimasd_vals = {p: [] for p in pops}
    sfs_sum = {p: None for p in pops}

    for i in range(N_REPLICATES):
        ts = msprime.sim_ancestry(
            samples={"AFR": N_SAMPLES, "EAS": N_SAMPLES, "AMR": N_SAMPLES},
            demography=d,
            sequence_length=SEQ_LEN,
            recombination_rate=RECOMB_RATE,
            random_seed=1000 + i,
        )
        mts = msprime.sim_mutations(ts, rate=MUT_RATE, random_seed=9000 + i)

        for p in pops:
            nodes = mts.samples(population=pop_ids[p])
            pi_vals[p].append(mts.diversity(sample_sets=nodes))
            tajimasd_vals[p].append(mts.Tajimas_D(sample_sets=nodes))

            # Folded SFS (polarised=False: we don't assume an ancestral-
            # allele call, just "minor allele frequency" bins), summed
            # across replicates below to get a less noisy average shape.
            sfs = mts.allele_frequency_spectrum(
                sample_sets=[nodes], polarised=False, span_normalise=False
            )
            sfs_sum[p] = np.array(sfs, dtype=float) if sfs_sum[p] is None \
                else sfs_sum[p] + np.array(sfs)

    summary = {}
    for p in pops:
        summary[p] = {
            "pi_mean": statistics.mean(pi_vals[p]),
            "pi_std": statistics.stdev(pi_vals[p]),
            "TajimasD_mean": statistics.mean(tajimasd_vals[p]),
            "TajimasD_std": statistics.stdev(tajimasd_vals[p]),
            "sfs_mean": (sfs_sum[p] / N_REPLICATES).tolist(),
        }
    return summary


print("Running baseline (bottleneck, no growth) - 20 replicates...")
summary_baseline = run_replicates(with_growth=False)
print("Running with post-bottleneck exponential growth - 20 replicates...")
summary_growth = run_replicates(with_growth=True)

print(f"\nGrowth rates used: r_EAS = {R_EAS:.6f}/gen   r_AMR = {R_AMR:.6f}/gen\n")
print(f"{'Pop':<5}{'pi (no growth)':<22}{'pi (growth)':<22}"
      f"{'Tajima D (no growth)':<24}{'Tajima D (growth)'}")
for p in ["AFR", "EAS", "AMR"]:
    b, g = summary_baseline[p], summary_growth[p]
    print(f"{p:<5}"
          f"{b['pi_mean']:.6f} +/- {b['pi_std']:.6f}   "
          f"{g['pi_mean']:.6f} +/- {g['pi_std']:.6f}   "
          f"{b['TajimasD_mean']:+.4f} +/- {b['TajimasD_std']:.4f}    "
          f"{g['TajimasD_mean']:+.4f} +/- {g['TajimasD_std']:.4f}")

# ------------------------------------------------------------------------
# 4. Save results (mean +/- std per statistic, plus the averaged SFS) and
#    append a summary line to the project's running comparison log.
# ------------------------------------------------------------------------
with open("v6_growth_results.json", "w") as f:
    json.dump({
        "params": {"T_AFR_EAS": T_AFR_EAS, "T_EAS_AMR": T_EAS_AMR,
                   "R_EAS": R_EAS, "R_AMR": R_AMR,
                   "MIG_RATE": MIG_RATE, "N_REPLICATES": N_REPLICATES},
        "baseline": summary_baseline,
        "growth": summary_growth,
    }, f, indent=2)
print("\nSaved v6_growth_results.json")

with open("model_comparison_stats.jsonl", "a") as f:
    for label, summary in [("v6_baseline_no_growth", summary_baseline),
                            ("v6_with_growth", summary_growth)]:
        row = {"model_version": label}
        for p in ["AFR", "EAS", "AMR"]:
            row[f"diversity_{p}_mean"] = summary[p]["pi_mean"]
            row[f"diversity_{p}_std"] = summary[p]["pi_std"]
            row[f"TajimasD_{p}_mean"] = summary[p]["TajimasD_mean"]
            row[f"TajimasD_{p}_std"] = summary[p]["TajimasD_std"]
        f.write(json.dumps(row) + "\n")
print("Appended v6_baseline_no_growth and v6_with_growth to model_comparison_stats.jsonl")

# ------------------------------------------------------------------------
# 5. Plot the folded SFS for each population, baseline vs. growth
# ------------------------------------------------------------------------
# Colorblind-safe 3-color palette (validated for this project: blue/orange/
# aqua pass all-pairs contrast checks for deuteranopia/protanopia/
# tritanopia, as used in earlier project figures).
COLORS = {"AFR": "#2a78d6", "EAS": "#eb6834", "AMR": "#1baf7a"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)
fig.patch.set_facecolor("#fcfcfb")

for ax, (label, summary) in zip(axes, [("No growth (bottleneck held flat)", summary_baseline),
                                        ("With post-bottleneck growth", summary_growth)]):
    ax.set_facecolor("#fcfcfb")
    for p in ["AFR", "EAS", "AMR"]:
        sfs = np.array(summary[p]["sfs_mean"])
        # Bin 0 is invariant (monomorphic) sites - not part of the allele-
        # frequency shape, so it's excluded from the plot. The remaining
        # bins are minor-allele-count classes 1..(2*N_SAMPLES - 1) for this
        # folded, diploid SFS; frequency 1 ("singletons") is the rarest
        # class and the one most sensitive to recent growth.
        freq_classes = np.arange(1, len(sfs))
        counts = sfs[1:]
        ax.plot(freq_classes, counts, color=COLORS[p], marker="o",
                 markersize=3, linewidth=1.4, label=p, alpha=0.9)
    ax.set_xlim(1, 20)  # rare/low-frequency end is what matters here
    ax.set_xlabel("Minor allele count (folded)")
    ax.set_ylabel("Mean number of sites (20 replicates)")
    ax.set_title(label, fontsize=11)
    ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.8)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.legend(frameon=False)

fig.suptitle("Folded site-frequency spectrum: bottleneck alone vs. bottleneck + growth",
             fontsize=13)
fig.tight_layout()
fig.savefig("sfs_comparison_v6.png", facecolor=fig.get_facecolor())
plt.close(fig)
print("Saved sfs_comparison_v6.png")

# Also report the singleton fraction directly - the single clearest number
# for "does this population have relatively more or fewer rare variants".
print("\nSingleton fraction of all segregating sites (higher = more rare variants):")
for label, summary in [("no growth", summary_baseline), ("with growth", summary_growth)]:
    for p in ["AFR", "EAS", "AMR"]:
        sfs = np.array(summary[p]["sfs_mean"])
        poly = sfs[1:]
        frac = poly[0] / poly.sum()
        print(f"  {label:<12} {p}: {frac:.3f}")

Running baseline (bottleneck, no growth) - 20 replicates...
Running with post-bottleneck exponential growth - 20 replicates...

Growth rates used: r_EAS = 0.001167/gen   r_AMR = 0.004171/gen

Pop  pi (no growth)        pi (growth)           Tajima D (no growth)    Tajima D (growth)
AFR  0.001921 +/- 0.000048   0.001916 +/- 0.000055   +0.0131 +/- 0.0598    +0.0040 +/- 0.0829
EAS  0.001391 +/- 0.000103   0.001643 +/- 0.000087   +2.0413 +/- 0.3036    +1.3220 +/- 0.1156
AMR  0.001355 +/- 0.000114   0.001597 +/- 0.000077   +2.0137 +/- 0.1986    +1.4352 +/- 0.1544

Saved v6_growth_results.json
Appended v6_baseline_no_growth and v6_with_growth to model_comparison_stats.jsonl
Saved sfs_comparison_v6.png

Singleton fraction of all segregating sites (higher = more rare variants):
  no growth    AFR: 0.191
  no growth    EAS: 0.045
  no growth    AMR: 0.038
  with growth  AFR: 0.196
  with growth  EAS: 0.078
  with growth  AMR: 0.078


In [20]:
# ------------------------------------------------------------------------
#    DIVERSITY AND TAJIMA'S D PLOT ----> NOT THIS ONE, IT'S IN BARPLOT
# 1. Load results
# ------------------------------------------------------------------------
RESULTS_PATH = "v6_growth_results.json"
with open(RESULTS_PATH) as f:
    results = json.load(f)
 
POPULATIONS = ["AFR", "EAS", "AMR"]
 
# Same population -> color mapping used throughout this project (validated
# colorblind-safe for all pairwise comparisons: blue / orange / aqua).
COLORS = {"AFR": "#2a78d6", "EAS": "#eb6834", "AMR": "#1baf7a"}
 
# ------------------------------------------------------------------------
# 2. Pull out the numbers we need
# ------------------------------------------------------------------------
def get_stat(variant, stat):
    return [results[variant][p][f"{stat}_mean"] for p in POPULATIONS]
 
 
def get_err(variant, stat):
    return [results[variant][p][f"{stat}_std"] for p in POPULATIONS]
 
 
pi_baseline = get_stat("baseline", "pi")
pi_growth = get_stat("growth", "pi")
pi_baseline_err = get_err("baseline", "pi")
pi_growth_err = get_err("growth", "pi")
 
tD_baseline = get_stat("baseline", "TajimasD")
tD_growth = get_stat("growth", "TajimasD")
tD_baseline_err = get_err("baseline", "TajimasD")
tD_growth_err = get_err("growth", "TajimasD")
 
# ------------------------------------------------------------------------
# 3. Draw the figure
# ------------------------------------------------------------------------
fig, (ax_pi, ax_td) = plt.subplots(1, 2, figsize=(12, 5.5), dpi=150)
fig.patch.set_facecolor("#fcfcfb")
 
x = np.arange(len(POPULATIONS))
bar_width = 0.32
 
 
def draw_panel(ax, baseline_vals, growth_vals, baseline_err, growth_err, ylabel, title):
    ax.set_facecolor("#fcfcfb")
 
    bars_baseline = ax.bar(
        x - bar_width / 2, baseline_vals, bar_width,
        yerr=baseline_err, capsize=4,
        color=[COLORS[p] for p in POPULATIONS],
        edgecolor="white", linewidth=0.8,
        error_kw={"ecolor": "#52514e", "elinewidth": 1.1},
        label="_nolegend_",
    )
    bars_growth = ax.bar(
        x + bar_width / 2, growth_vals, bar_width,
        yerr=growth_err, capsize=4,
        color=[COLORS[p] for p in POPULATIONS],
        edgecolor="white", linewidth=0.8,
        hatch="///",
        error_kw={"ecolor": "#52514e", "elinewidth": 1.1},
        label="_nolegend_",
    )
 
    ax.set_xticks(x)
    ax.set_xticklabels(POPULATIONS)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, pad=10)
    ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color("#c3c2b7")
    ax.tick_params(colors="#52514e")
    return bars_baseline, bars_growth
 
 
draw_panel(ax_pi, pi_baseline, pi_growth, pi_baseline_err, pi_growth_err,
           "Nucleotide diversity (π)", "Nucleotide diversity")
 
draw_panel(ax_td, tD_baseline, tD_growth, tD_baseline_err, tD_growth_err,
           "Tajima's D", "Tajima's D")
ax_td.axhline(0, color="#52514e", linewidth=1, linestyle="--", zorder=1)
 
# ------------------------------------------------------------------------
# 4. Legends: population (color) + growth status (fill pattern)
# ------------------------------------------------------------------------
from matplotlib.patches import Patch
 
population_handles = [Patch(facecolor=COLORS[p], edgecolor="white", label=p)
                       for p in POPULATIONS]
growth_handles = [
    Patch(facecolor="#9b9a92", edgecolor="white", label="No growth"),
    Patch(facecolor="#9b9a92", edgecolor="white", hatch="///", label="With growth"),
]
 
fig.legend(handles=population_handles, loc="upper center",
           bbox_to_anchor=(0.27, 0.0), ncol=3, frameon=False, title="Population")
fig.legend(handles=growth_handles, loc="upper center",
           bbox_to_anchor=(0.73, 0.0), ncol=2, frameon=False, title="Variant")
 
fig.suptitle("Diversity and Tajima's D: post-bottleneck growth vs. no growth",
             fontsize=13.5, y=1.02)
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig("pi_tajimasd_comparison_v6.png", facecolor=fig.get_facecolor(),
            bbox_inches="tight")
plt.close(fig)
print("Saved pi_tajimasd_comparison_v6.png")


Saved pi_tajimasd_comparison_v6.png


In [25]:
# ------------------------------------------------------------------------
#    VIOLIN PLOT FOR NUCLEOTIDE DIVERSITY AND TAJIMA'S D

POPULATIONS = ["AFR", "EAS", "AMR"]
COLORS = {"AFR": "#2a78d6", "EAS": "#eb6834", "AMR": "#1baf7a"}
RAW_VALUES_PATH = "v6_growth_raw_values.json"
 
# ------------------------------------------------------------------------
# 1. Simulate the WITH-GROWTH demography and save the raw per-replicate
#    values (same parameters/seeds as native_american_demography_v6_growth.py)
# ------------------------------------------------------------------------
GEN_TIME = 29
T_AFR_EAS = round(40_000 / GEN_TIME)   # 1379 generations
T_EAS_AMR = round(16_000 / GEN_TIME)   # 551 generations
 
N_AFR = 40_000
 
N_EAS_FOUND = 2_000
N_EAS_NOW = 10_000
R_EAS = math.log(N_EAS_NOW / N_EAS_FOUND) / T_AFR_EAS
 
N_AMR_FOUND = 1_791
N_AMR_NOW = 17_910
R_AMR = math.log(N_AMR_NOW / N_AMR_FOUND) / T_EAS_AMR
 
MIG_RATE = 1e-7         # see growth_v6_README.md for why this rate is used
SEQ_LEN = 1e6
RECOMB_RATE = 1e-8
MUT_RATE = 1.2e-8
N_SAMPLES = 50
N_REPLICATES = 20
 
 
def build_demography_with_growth():
    d = msprime.Demography()
    d.add_population(name="AFR", initial_size=N_AFR)
    d.add_population(name="EAS", initial_size=N_EAS_NOW, growth_rate=R_EAS)
    d.add_population(name="AMR", initial_size=N_AMR_NOW, growth_rate=R_AMR)
    d.add_population_split(time=T_EAS_AMR, derived=["AMR"], ancestral="EAS")
    d.add_population_split(time=T_AFR_EAS, derived=["EAS"], ancestral="AFR")
    d.set_symmetric_migration_rate(["AFR", "EAS"], rate=MIG_RATE)
    d.sort_events()
    return d
 
 
def simulate_raw_values():
    d = build_demography_with_growth()
    pop_ids = {p: [pp.id for pp in d.populations if pp.name == p][0] for p in POPULATIONS}
 
    raw = {f"diversity_{p}": [] for p in POPULATIONS}
    raw.update({f"TajimasD_{p}": [] for p in POPULATIONS})
 
    for i in range(N_REPLICATES):
        ts = msprime.sim_ancestry(
            samples={"AFR": N_SAMPLES, "EAS": N_SAMPLES, "AMR": N_SAMPLES},
            demography=d,
            sequence_length=SEQ_LEN,
            recombination_rate=RECOMB_RATE,
            random_seed=1000 + i,
        )
        mts = msprime.sim_mutations(ts, rate=MUT_RATE, random_seed=9000 + i)
        for p in POPULATIONS:
            nodes = mts.samples(population=pop_ids[p])
            raw[f"diversity_{p}"].append(mts.diversity(sample_sets=nodes))
            raw[f"TajimasD_{p}"].append(mts.Tajimas_D(sample_sets=nodes))
 
    with open(RAW_VALUES_PATH, "w") as f:
        json.dump(raw, f, indent=2)
    print(f"Saved {RAW_VALUES_PATH}")
    return raw
 
 
print(f"Simulating {N_REPLICATES} replicates of the with-growth demography...")
raw = simulate_raw_values()
 
 
# ------------------------------------------------------------------------
# 2. Reusable violin-plot function
# ------------------------------------------------------------------------
# (raw values are now in memory - this loads the just-saved file back in
# too, purely so this section works unchanged if run on its own later)
with open(RAW_VALUES_PATH) as f:
    raw = json.load(f)
 
def plot_violin(key_prefix, ylabel, title, out_path, zero_line=False):
    data = [np.array(raw[f"{key_prefix}_{p}"]) for p in POPULATIONS]
 
    fig, ax = plt.subplots(figsize=(7, 5.5), dpi=150)
    fig.patch.set_facecolor("#fcfcfb")
    ax.set_facecolor("#fcfcfb")
 
    positions = np.arange(1, len(POPULATIONS) + 1)
    parts = ax.violinplot(
        data, positions=positions, showmeans=True, showmedians=False,
        showextrema=False, widths=0.75,
    )
 
    # Color each violin body by its population, with a soft fill and a
    # solid edge in the same hue - identity carried by color, consistent
    # with every other figure in this project.
    for body, pop in zip(parts["bodies"], POPULATIONS):
        body.set_facecolor(COLORS[pop])
        body.set_edgecolor(COLORS[pop])
        body.set_alpha(0.35)
        body.set_linewidth(1.5)
 
    # The mean marker msprime draws by default is a thin line - restyle it
    # to match each population's color and make it visible.
    parts["cmeans"].set_color([COLORS[p] for p in POPULATIONS])
    parts["cmeans"].set_linewidth(2)
 
    # Overlay the individual 20 replicate values as jittered points, so the
    # actual data is visible underneath the smoothed violin shape.
    rng = np.random.default_rng(0)
    for pos, pop, vals in zip(positions, POPULATIONS, data):
        jitter = rng.uniform(-0.12, 0.12, size=len(vals))
        ax.scatter(
            pos + jitter, vals, color=COLORS[pop], s=22,
            edgecolors="white", linewidths=0.6, alpha=0.9, zorder=3,
        )
 
    if zero_line:
        ax.axhline(0, color="#52514e", linewidth=1, linestyle="--", zorder=1)
 
    ax.set_xticks(positions)
    ax.set_xticklabels(POPULATIONS)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12.5, pad=12)
    ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color("#c3c2b7")
    ax.tick_params(colors="#52514e")
 
    fig.tight_layout()
    fig.savefig(out_path, facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"Saved: {out_path}")
 
 
# ------------------------------------------------------------------------
# 3. Make the two figures
# ------------------------------------------------------------------------
plot_violin(
    key_prefix="diversity",
    ylabel="Nucleotide diversity (π)",
    title="Nucleotide diversity across 20 replicates (with growth)",
    out_path="violin_diversity_growth_v6.png",
)
 
plot_violin(
    key_prefix="TajimasD",
    ylabel="Tajima's D",
    title="Tajima's D across 20 replicates (with growth)",
    out_path="violin_tajimasd_growth_v6.png",
    zero_line=True,
)


Simulating 20 replicates of the with-growth demography...
Saved v6_growth_raw_values.json
Saved: violin_diversity_growth_v6.png
Saved: violin_tajimasd_growth_v6.png
